

Trained a model that seeks to perform classification of medical OCT scans. In the data there are four classes of scans: Healthy, CNV, DME and DRUSEN. The last three are eye diseases that cause visible damage to the retina and can be spotted through the OCT scans.



In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torchvision import utils
import torch.nn.init as init
import pandas as pd
import io
from PIL import Image
import time
import os
import copy

import torchvision


import matplotlib.pyplot as plt
import numpy as np


path_to_data = #path to where you downloaded the raw data
path_train = path_to_data + 'train'
path_test = path_to_data + 'test'
path_val = path_to_data + 'val'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

**First Step:** Load the Data, setup Transforms, and Initialize the Dataloader


[Here](https://pytorch.org/vision/stable/transforms.html) is a great resource to learn more about the different transforms that can be added. The goal of the transform is to properly prepare the data to be sent to the model and to add data augmentation. You may have pictures of different resolution sizes, so here is a good time to set a transform to make the sizes of images uniform.

In [ ]:

# Transforms for train set
train_transform = transforms.Compose(
      [transforms.ToTensor(),   transforms.RandomHorizontalFlip(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])                                # Add Transforms


# Transforms for test/val set
test_val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

In [ ]:
batchsize = 64 # decide your batch size. REMEMBER: If your batch size is too big, you won't be able to load the data to the GPU


# dataset_train = datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform)
# dataset_test = datasets.CIFAR10(root='./data', train=False, download=True, transform=test_val_transform)

#Initialize dataloader for train set
dataset_train = #code dataset
train_loader = torch.utils.data.DataLoader(dataset_train, batch_size = batchsize, shuffle=True)

#Initialize dataloader for test set
dataset_test = #code dataset
test_loader = torch.utils.data.DataLoader(dataset_test, batch_size = batchsize, shuffle=False)

#Initialize dataloader for val set
dataset_val = #code dataset
val_loader = torch.utils.data.DataLoader(dataset_val, batch_size = batchsize, shuffle=True)

**Second Step:** Design Model's Architecture and code it here in with PyTorch.



In [ ]:
# Choose model architecture
model =


**Third Step:** Code Fit and Test functions. This is similar to Lab 3, but this time make sure to use the validation set as well.

In [ ]:
# Code fit function
def fit():



# Code test function
def test_accuracy(model, test_loader, device, stat_count=100):
        test_losses = []
        model.to(device)
        correct = 0
        total = 0

        with torch.no_grad():
            for test_data in test_loader:
                #images, labels = test_data[0].cuda(), test_data[1].cuda()
                images, labels = test_data['images'].to(device), test_data['labels'].to(device)
                # images = images.view(-1, input_size)
                outputs = model(images)
                loss = loss_function(outputs, labels)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
                accuracy = 100 * correct / total

                test_losses.append(loss.item())
        test_loss = sum(test_losses)/len(train_loader)

        return test_loss, accuracy


**Fourth Step:** Set Parameters and run model.


In [ ]:
torch.manual_seed(0)


##Set up parameters
input_size =
num_classes =
num_epochs =
lr =

#Initialize model and send it to cuda
net = model
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
net.to(device)

#Set loss function and optimizer
loss_function = nn.CrossEntropyLoss()

# Optimizer
optimizer = torch.optim.Adam(resnet18.parameters(), lr=lr)

#perform fit

#perform test


##### LAST STEP, SAVE MODEL
path_model_save = #choose path to save model
torch.save(net.state_dict(), path_model_save)

**Fifth Step:** Model interpretability.

For this assignment, you will interpret the model's results through the use of saliency mapping. You will use the following package: [GitHub](https://github.com/jacobgil/pytorch-grad-cam).

You are expected to install the package on your environment and go through the GitHub to learn its application. Below is an example code to help you get started:

In [ ]:
### Sample Use with ResNet50:

from pytorch_grad_cam import GradCAM, HiResCAM, ScoreCAM, GradCAMPlusPlus, AblationCAM, XGradCAM, EigenCAM, FullGrad
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
from torchvision.models import resnet50

model = resnet50(pretrained=True)
target_layers = [model.layer4[-1]]


input_tensor = # Your input data


# Note: input_tensor can be a batch tensor with several images!

# Construct the CAM object once, and then re-use it on many images:
cam = GradCAM(model=model, target_layers=target_layers, use_cuda=False)

targets = ### your label

# You can also pass aug_smooth=True and eigen_smooth=True, to apply smoothing.
grayscale_cam = cam(input_tensor=input_tensor, targets=targets)

# In this example grayscale_cam has only one image in the batch:
grayscale_cam = grayscale_cam[0, :]
visualization = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)